## Setup


In [7]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


df = pd.read_csv("../data/processed/monthly_returns.csv")
monthly = df.pivot(index="date", columns="series", values="return_pct")



## Chart style

Both charts below share one registered plotly template, so the house style is
defined once instead of being repeated per figure.

In [8]:

# Colours and fonts
MARKET, SAVINGS = "#0f5499", "#990f3d"
PAPER = "#fffaf3"                       # warm off-white, as used in print
INK, MUTED, GRID = "#16232e", "#6b6459", "#e5ded4"
SERIF = "Georgia, Times New Roman, serif"
SANS = "Inter, -apple-system, Segoe UI, Helvetica, Arial, sans-serif"


# One registered template instead of repeating the styling in each figure, so the
# two charts cannot drift apart.
pio.templates["me204"] = go.layout.Template(
    layout=dict(
        colorway=[MARKET, SAVINGS],
        font=dict(family=SANS, size=13, color=INK),
        title=dict(font=dict(family=SERIF, size=21, color=INK),
                   x=0.5, xanchor="center", xref="container", pad=dict(b=16)),
        paper_bgcolor=PAPER,
        plot_bgcolor=PAPER,
        # No axis lines: the gridlines and the baseline carry the structure.
        xaxis=dict(showgrid=False, showline=True, linecolor=INK, linewidth=1.2,
                   ticks="outside", tickcolor=GRID, ticklen=4,
                   tickfont=dict(size=12, color=MUTED),
                   title=dict(font=dict(size=11.5, color=MUTED))),
        yaxis=dict(side="right", showgrid=True, gridcolor=GRID, gridwidth=1,
                   zeroline=False, showline=False, ticks="",
                   tickfont=dict(size=12, color=MUTED),
                   title=dict(font=dict(size=11.5, color=MUTED))),
        legend=dict(orientation="h", y=1.0, x=0, yanchor="bottom",
                    font=dict(size=12, color=MUTED)),
        hoverlabel=dict(bgcolor="white", bordercolor=GRID,
                        font=dict(family=SANS, size=12, color=INK)),
        margin=dict(l=25, r=80, t=115, b=70),
    )
)

# "+" composes them: plotly_white is the base, me204 overrides what it defines.
pio.templates.default = "plotly_white+me204"

SOURCE = ("Source: Federal Reserve Economic Data (FRED), series SP500 and "
          "BRMSA0104. Data retrieved 22 July 2026.")


def press_frame(fig, source=SOURCE):
    """Put the source line in the footer. Paper coordinates cover the plot area,
    not the image, so y is derived from the margins and survives a height change."""
    plot_h = fig.layout.height - fig.layout.margin.t - fig.layout.margin.b

    fig.add_annotation(xref="paper", yref="paper",
                       x=0.5, y=-(fig.layout.margin.b - 30) / plot_h,
                       xanchor="center", yanchor="top", showarrow=False,
                       text=source, font=dict(size=10.5, color=MUTED))


def style_subplot_titles(fig, size=14):
    """Restyle the panel titles make_subplots writes, which default to centred
    black sans and otherwise read as a second headline."""
    for note in fig.layout.annotations:
        note.update(x=0, xanchor="left",
                    font=dict(family=SERIF, size=size, color=MUTED))


def export(fig, name):
    """Write the figure to docs/ as HTML (keeps the hover) and PNG (fallback), so
    rerunning this notebook refreshes the website."""
    path = Path(f"../docs/{name}.html")
    fig.write_html(path, include_plotlyjs="cdn", full_html=True,
                   config={"displayModeBar": False, "responsive": True})

    # plotly's page keeps the browser's 8px body margin, which overflows the
    # iframe and raises scrollbars over the chart. Zeroing it makes the frame
    # height in the markdown exact.
    path.write_text(path.read_text().replace(
        "<head>",
        "<head><style>html,body{margin:0;padding:0;overflow:hidden}</style>", 1))

    fig.write_image(f"../docs/{name}.png", scale=2)

## Finding 1 - what $1,000 became



In [9]:
START = 1000    # invested once, in Aug 2016

# A 2% month is a multiplier of 1.02, so the running product of the multipliers
# is what the money was worth. Returns compound, they do not add up.
growth = (1 + monthly / 100).cumprod() * START

# A starting row, so both lines open level at $1,000 rather than after a month.
growth.loc["2016-07"] = START
growth = growth.sort_index()

x = [d + "-01" for d in growth.index]   # plotly reads a date only with a day in it
final = growth.iloc[-1]   # last row: the two end values quoted on the website

fig = go.Figure()


# Shaded bands behind the lines, naming the two stretches the page argues about.
for x0, x1, label in [("2020-02-01", "2020-08-01", "COVID crash<br>and recovery"),
                      ("2021-12-01", "2022-10-01", "2022<br>drawdown")]:
    fig.add_vrect(x0=x0, x1=x1, fillcolor="#f0e9dd", line_width=0, layer="below",
                  annotation_text=label, annotation_position="top left",
                  annotation_font=dict(size=10, color=MUTED))

for name, col, color, width in [("S&P 500", "SP500", MARKET, 2.4),
                                ("Savings account", "BRMSA0104", SAVINGS, 2)]:
    fig.add_trace(go.Scatter(
        x=x, y=growth[col], name=name, mode="lines",
        line=dict(color=color, width=width),
        # %{x} and %{y} are the hovered point's values; the part after the pipe
        # formats them, and <extra> replaces plotly's default trace-name box.
        hovertemplate="%{x|%b %Y}: $%{y:,.0f}<extra>" + name + "</extra>",
    ))

    # xref="paper" is a fraction of the plot area, so 1.075 sits just outside it.
    fig.add_annotation(
        xref="paper", x=1.075, y=final[col],
        text=f"<b>${final[col]:,.0f}</b><br>{name}",
        xanchor="left", align="left", showarrow=False,
        font=dict(color=color, size=12.5),
    )

fig.update_layout(
    title=dict(
        text=f"${START:,} invested in Aug 2016, and what it was worth by Jul 2026",
        # &#36; not $ - plotly treats a $...$ pair as LaTeX and would eat the text.
        subtitle=dict(
            text=f"The S&P 500 ended at &#36;{final['SP500']:,.0f}, the savings "
                 f"account at &#36;{final['BRMSA0104']:,.0f} \u2014 a gap of "
                 f"&#36;{final['SP500'] - final['BRMSA0104']:,.0f}.",
            font=dict(family=SANS, size=13, color=MUTED)),
    ),
    height=520, showlegend=False, hovermode="x unified",
    margin=dict(l=25, r=175, t=160, b=75),
)
fig.update_yaxes(title_text=None, tickprefix="$", tickformat=",.0f")
fig.update_xaxes(title_text=None, range=[x[0], x[-1]], dtick="M24", tickformat="%Y")

press_frame(fig)

export(fig, "finding1-growth")

fig.show()

## Finding 2 - year by year



In [10]:
annual = monthly.copy()
annual["year"] = [d[:4] for d in annual.index]   # "2018-03" -> "2018"

# 2016 holds five months and 2026 seven, so neither is comparable with a full
# year. Nine complete calendar years remain.
annual = annual.loc[~annual["year"].isin(["2016", "2026"])]

# One group per year, months compounded inside it, results stacked back into a
# table. include_groups=False keeps the text column out of the arithmetic.
yearly = annual.groupby("year").apply(
    lambda g: (1 + g[["SP500", "BRMSA0104"]] / 100).prod() * 100 - 100,
    include_groups=False,
)

years = yearly.index.tolist()
won = (yearly["SP500"] > yearly["BRMSA0104"]).sum()
lost = [y for y in years if yearly.loc[y, "SP500"] < yearly.loc[y, "BRMSA0104"]]

# Compound annual rate, not the mean of the nine figures: averaging returns
# overstates them, since recovering -16.31% takes more than 16.31%. This is also
# the rate that reconciles with finding 1.
rate = lambda col: (((1 + yearly[col] / 100).prod()) ** (1 / len(years)) - 1) * 100
compound, compound_savings = rate("SP500"), rate("BRMSA0104")

print(f"compound annual return over {len(years)} years - "
      f"market {compound:.2f}%, savings {compound_savings:.3f}%")

fig = go.Figure()


fig.add_vline(x=compound, line=dict(color=MARKET, width=1.2, dash="dot"),
              annotation_text=f"compound average {compound:.1f}% a year",
              annotation_position="bottom right",
              annotation_font=dict(size=10.5, color=MARKET))

# One trace with None between each pair, rather than nine separate shapes -
# shapes do not place reliably on a category axis.
link_x, link_y = [], []
for yr in years:
    link_x += [yearly.loc[yr, "BRMSA0104"], yearly.loc[yr, "SP500"], None]
    link_y += [yr, yr, None]

fig.add_trace(go.Scatter(x=link_x, y=link_y, mode="lines", hoverinfo="skip",
                         line=dict(color="#ddd6cb", width=3), showlegend=False))

for name, col, color in [("Savings account", "BRMSA0104", SAVINGS),
                         ("S&P 500", "SP500", MARKET)]:
    fig.add_trace(go.Scatter(
        x=yearly[col], y=years, mode="markers", name=name,
        marker=dict(color=color, size=13, line=dict(color=PAPER, width=2)),
        hovertemplate="%{y}: %{x:.2f}%<extra>" + name + "</extra>",
    ))


# Series named above the top row instead of in a legend. Annotations on a
# category axis are placed by position, and the axis runs bottom-up.
top = len(years) - 1
for name, col, color in [("S&P 500", "SP500", MARKET),
                         ("Savings account", "BRMSA0104", SAVINGS)]:
    fig.add_annotation(x=yearly.loc[years[-1], col], y=top, text=name,
                       showarrow=False, yshift=24,
                       font=dict(color=color, size=12))


for yr in lost:
    fig.add_annotation(x=yearly.loc[yr, "BRMSA0104"], y=years.index(yr),
                       text="savings ahead", showarrow=False,
                       xshift=14, xanchor="left",
                       font=dict(size=10, color=SAVINGS))

fig.update_layout(
    title=dict(
        text=f"Return by calendar year: the market was ahead in {won} of {len(years)} years",
        subtitle=dict(
            text="Each line is one year. Its length is what choosing the market "
                 "was worth that year - or, twice, what it cost.",
            font=dict(family=SANS, size=13, color=MUTED)),
    ),
    height=560, showlegend=False,
    margin=dict(l=25, r=45, t=160, b=105),
)


# type="category" is required: otherwise plotly reads "2017" as a date and lays
# the axis out as a timeline, which loses the rows.
fig.update_yaxes(type="category", side="left",
                 showgrid=False, ticks="", title_text=None,
                 tickfont=dict(size=13, color=MUTED))
fig.update_xaxes(title_text="Return over the year (%)", ticksuffix="%",
                 showgrid=True, gridcolor=GRID, showline=False,
                 zeroline=True, zerolinecolor="#a8a095", zerolinewidth=1.2)

press_frame(fig)
export(fig, "finding2-by-year")

fig.show()


compound annual return over 9 years - market 13.19%, savings 0.212%
